In [ ]:
from pathlib import Path
import os

from natsort import natsorted
from tifffile import imread
from skimage.measure import regionprops_table
import pandas as pd
import numpy as np

def mode(arr):
    vals, counts = np.unique_counts(arr)
    return vals[np.argmax(counts)].item()

image_props_funs = {
    'mode': mode,
    'mean': np.mean,
}

In [ ]:
in_path = '/home/hoerl/tetraploid-nn/Samriddhi/14-07-26/3CS_Human_NO_DOX'

segmentations = [
    {'name': 'nucleus',
        'subdirectory': 'segmentation_nuclei',
        'file_pattern': '[!.]*.tif'
    },

    {'name': 'border',
        'subdirectory': 'segmentation_nuclei_border',
        'file_pattern': '[!.]*.tif'
    }
]

intensity_images = [
    {'name': 'uhrf1',
        'subdirectory': 'tif',
        'file_pattern': '[!.]*ch0.tif'
    },

    {'name': 'dppa3',
        'subdirectory': 'tif',
        'file_pattern': '[!.]*ch1.tif'
    }
]

out_subdirectory = 'region_properties_multichannel'

shape_properties_to_include = ('label', 'area', )
intensity_properties_to_include = ('intensity_mean', )
image_wide_properties = ('mode', )




In [ ]:
mask_files = []
for segmentation_spec in segmentations:
    mask_files_i = natsorted((Path(in_path) / segmentation_spec['subdirectory']).glob(segmentation_spec['file_pattern']))
    mask_files.append(mask_files_i)

image_files = []
for image_file_spec in intensity_images:
    image_files_i = natsorted((Path(in_path) / image_file_spec['subdirectory']).glob(image_file_spec['file_pattern']))
    image_files.append(image_files_i)

In [ ]:
if 'label' not in shape_properties_to_include:
    shape_properties_to_include = ('label', ) + shape_properties_to_include

if 'label' not in intensity_properties_to_include:
    intensity_properties_to_include = ('label', ) + intensity_properties_to_include

outdir = Path(in_path) / out_subdirectory

if not outdir.exists():
    outdir.mkdir()
    

def process_one(mask_files_i, image_files_i):

    file_prefix = os.path.commonprefix([f.name for f in mask_files_i] + [f.name for f in image_files_i]).removesuffix('_ch')
    
    df = pd.DataFrame({p: [] for p in shape_properties_to_include + intensity_properties_to_include})
    
    for segmentation_idx, mask_file_ij in enumerate(mask_files_i):
    
        seg_name = segmentations[segmentation_idx]["name"]
        
        mask = imread(mask_file_ij)
    
        dfi = pd.DataFrame(regionprops_table(mask, properties=shape_properties_to_include))
    
        df = pd.merge(df, dfi, on='label', how='outer', suffixes=('', f'_{seg_name}'))
    
    
        for channel_idx, image_file_ij in enumerate(image_files_i):
    
            img = imread(image_file_ij)
            ch_name = intensity_images[channel_idx]["name"]
    
            dfi = pd.DataFrame(regionprops_table(mask, img, properties=intensity_properties_to_include))
            df = pd.merge(df, dfi, on='label', how='outer', suffixes=('', f'_{seg_name}_{ch_name}'))
    
    
            for prop in image_wide_properties:
    
                col_name = f'{prop}_{ch_name}'
                if col_name not in df.columns:
    
                    df[col_name] = image_props_funs[prop](img)
            
    
    df = df.dropna(axis=1)
    df.to_csv(outdir / f'{file_prefix}_measurements.csv', index=None)


In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor() as tpe:

    futures = []
    for mask_files_i, image_files_i in zip(zip(*mask_files), zip(*image_files)):
        futures.append(tpe.submit(process_one, mask_files_i, image_files_i))

    for f in futures:
        f.result()